# GNN imitation training (run 11, phase 2)

Thesis figures for one variant of the run-11 GNN grid. The notebook reads the run from the folder
it sits in (`diagrams/data/<run name>/`), so the same notebook serves all eight variants. To
compare variants in one diagram, use `diagrams/gnn_comparison.ipynb`.

Each GNN is trained to output the target paper's own Voyage embedding from the papers it cites
(the target row is masked). The objective is $1-\cos$ to that embedding, optionally plus
$0.05\cdot$InfoNCE ($\tau=0.05$, 4,096 random negatives). After every epoch it is measured on
the 806 validation papers. The baseline is the plain mean of the references' embeddings,
measured once before training on the same papers.

There is one figure per W&B metric (plus the training objective), each in its own file under
`figures/`, as PDF for LaTeX and PNG for previews. Figures are drawn at their printed size, so
include them without a `width=` option.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, PercentFormatter

RUN_DIR = Path.cwd()                          # diagrams/data/<run name>/
sys.path.insert(0, str(RUN_DIR.parents[1]))   # diagrams/, for thesis_style
import thesis_style as ts

ts.apply()
# Printed size of every figure: 0.75 \textwidth matches the template's caption width.
# ts.TEXTWIDTH = full width, ts.HALFWIDTH = two subfigures side by side.
ts.FIGWIDTH, ts.FIGHEIGHT = 0.75 * ts.TEXTWIDTH, 70 * ts.MM
C = ts.COLORS
OUT = RUN_DIR / "figures"

RUN = RUN_DIR.name
SHORT = RUN.removeprefix("run11-").removesuffix("-14541449")   # e.g. gnn-gatv2-l1-cos
config = json.loads((RUN_DIR / "config.json").read_text())
history = pd.read_csv(RUN_DIR / "history.csv")

# The reference-mean baseline: one row, logged before the first epoch.
baseline = history.filter(like="pretrain/refs_mean_").dropna(how="all").iloc[0]
baseline.index = baseline.index.str.removeprefix("pretrain/refs_mean_")

log = history.dropna(subset=["pretrain/epoch"]).copy()
log.columns = log.columns.str.removeprefix("pretrain/")
log["epoch"] = log["epoch"].astype(int) + 1   # W&B has 0-based epochs

# The epoch pretrain_encoder.py kept: best validation cosine among the epochs whose output
# pairwise cosine is within the cosine gate (the gate is 1, i.e. off, in run 11).
eligible = log[log["output_cosine"] <= config["cosine_gate"]]
best = eligible.loc[eligible["mean_cosine"].idxmax()]

CONV = {"gatv2": "GATv2", "sage": "GraphSAGE"}[config["gnn_conv"]]
OBJECTIVE = ("cosine" if config["lambda_nce"] == 0
             else f"cosine + {config['lambda_nce']:g} x InfoNCE")
print(f"{RUN}\n  {config['gnn_layers']}-layer {CONV}, objective {OBJECTIVE}, "
      f"{len(log)} epochs, lr {config['lr']:g}, batch {config['batch_size']}")
print(f"  selected epoch {int(best.epoch)}: validation cosine {best.mean_cosine:.4f} "
      f"(reference mean {baseline.mean_cosine:.4f}, {best.mean_cosine - baseline.mean_cosine:+.4f}), "
      f"top-1 {best.top1:.1%} (reference mean {baseline.top1:.1%})")

RING = dict(ls="none", marker="o", ms=7, mfc="none", mec=C["black"], mew=0.9, zorder=3)

def plot_metric(key, ylabel, name, color=C["blue"], pct=False, with_baseline=True,
                with_selected=True, ylim=None):
    # One W&B metric over epochs: the GNN, the reference mean (dashed) and the selected epoch.
    scale = 100 if pct else 1
    fig, ax = ts.figure()
    ax.plot(log.epoch, scale * log[key], color=color, label="GNN")
    if with_baseline and key in baseline.index:
        ax.axhline(scale * baseline[key], color=C["grey"], ls="--", lw=1, label="Reference mean")
    if with_selected:
        ax.plot(best.epoch, scale * best[key], label="Selected epoch", **RING)
    ax.set(xlabel="Epoch", ylabel=ylabel, xlim=(0.5, log.epoch.max() + 0.5))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if pct:
        ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
    if ylim is not None:
        ax.set_ylim(*ylim)
    if len(ax.get_legend_handles_labels()[0]) > 1:
        ts.legend_below(fig)
    ts.save(fig, f"{SHORT}_{name}", OUT)
    plt.show()

## Validation cosine to the paper's own embedding

The quantity the GNN is trained on, measured on the 806 validation papers. The ring marks the
selected epoch: the best validation cosine.

*Suggested caption:* Cosine similarity between the GNN output and the paper's own embedding on
the validation papers ([variant]). Dashed: plain mean of the references' embeddings.

In [ ]:
plot_metric("mean_cosine", "Cosine to own embedding", "cosine_validation")

## Training cosine

The same cosine, averaged over the training batches of each epoch. It is lower than on
validation because the papers differ, not because of overfitting. Training papers have a median
of 5 references; validation papers have 45–52.

*Suggested caption:* Cosine similarity between GNN output and own embedding on training batches
([variant]).

In [ ]:
plot_metric("train_cosine", "Cosine to own embedding", "cosine_training", color=C["orange"],
            with_baseline=False, with_selected=False)

## Training objective

Built from the logged terms and the weights in the run's config:
$\lambda_{\cos}(1-\cos) + \lambda_{\mathrm{NCE}}\,\mathrm{InfoNCE}$. The VICReg weights are 0 in
run 11. For the InfoNCE variants its two weighted components are drawn as well.

*Suggested caption:* Training objective of the [variant] GNN.

In [ ]:
cos_term = config["lambda_cos"] * (1 - log.train_cosine)
nce_term = config["lambda_nce"] * log.loss
objective = (cos_term + nce_term + config["vicreg_var"] * log.variance
             + config["vicreg_cov"] * log.covariance)

fig, ax = ts.figure()
ax.plot(log.epoch, objective, color=C["black"], label="Objective")
if config["lambda_nce"] > 0:
    cos_name = r"$1-\cos$" if config["lambda_cos"] == 1 else rf"${config['lambda_cos']:g}\,(1-\cos)$"
    ax.plot(log.epoch, cos_term, color=C["blue"], ls="--", label=cos_name)
    ax.plot(log.epoch, nce_term, color=C["vermillion"], ls=":",
            label=rf"${config['lambda_nce']:g}\cdot\mathrm{{InfoNCE}}$")
    ts.legend_below(fig)
ax.set(xlabel="Epoch", ylabel="Training loss", xlim=(0.5, log.epoch.max() + 0.5),
       ylim=(0, 1.08 * objective.max()))
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ts.save(fig, f"{SHORT}_objective", OUT)
plt.show()

## Retrieval on the validation papers: top-1 and top-5

Each GNN output is ranked against the true embeddings of all 806 validation papers (chance
top-1 = 0.12%). Shown: the share of papers whose own embedding is ranked first, or within the
top five.

*Suggested caption:* Top-1 [top-5] retrieval of the paper's own embedding among 806 validation
papers ([variant]). Dashed: plain reference mean.

In [ ]:
plot_metric("top1", "Top-1 retrieval", "top1", pct=True)

In [ ]:
plot_metric("top5", "Top-5 retrieval", "top5", color=C["green"], pct=True)

## Median rank of the own embedding

The median position of the own embedding in the same ranking. It is 1 whenever more than half of
the papers are retrieved first, which is the case here from the first epoch.

*Suggested caption:* Median retrieval rank of the paper's own embedding among 806 validation
papers ([variant]).

In [ ]:
plot_metric("median_rank", "Median rank", "median_rank", color=C["purple"],
            ylim=(0, max(2, log.median_rank.max()) + 1))

## Output diversity

Mean pairwise cosine between the GNN outputs of different validation papers. A value near 1
would mean every paper gets the same vector. For scale: the reader's own-embedding input sits
at 0.378; the step-7 encoder, which collapsed, at 0.660.

*Suggested caption:* Mean pairwise cosine between the GNN outputs of different validation
papers ([variant]). Dashed: plain reference mean.

In [ ]:
plot_metric("output_cosine", "Output pairwise cosine", "output_cosine", color=C["purple"],
            ylim=(0, 1))

## InfoNCE loss and in-batch accuracy (training)

InfoNCE against 4,096 random negatives, and how often the positive scores highest in it. Both
are logged for every variant; in the cosine-only variants the InfoNCE weight is 0, so they are
monitored, not optimised.

*Suggested caption:* InfoNCE loss [in-batch accuracy] on training batches ([variant]).

In [ ]:
plot_metric("loss", "InfoNCE (training)", "infonce", color=C["vermillion"],
            with_baseline=False, with_selected=False)

In [ ]:
plot_metric("batch_accuracy", "In-batch accuracy (training)", "batch_accuracy",
            color=C["vermillion"], pct=True, with_baseline=False, with_selected=False)

## VICReg terms (training, diagnostics only)

The VICReg variance hinge and the off-diagonal covariance of the outputs, per batch. Their
weights are 0 in run 11, so they do not affect training.

*Suggested caption:* VICReg variance [covariance] term of the GNN outputs on training batches
([variant]); not part of the run-11 objective.

In [ ]:
plot_metric("variance", "Variance term", "vicreg_variance", color=C["sky"],
            with_baseline=False, with_selected=False)

In [ ]:
plot_metric("covariance", "Covariance term", "vicreg_covariance", color=C["sky"],
            with_baseline=False, with_selected=False)